# Paeonia workbench

This editable notebook moves from spelled pitches to score-wide tonal plans. It is created by `python -m paeonia`.

In [ ]:
from paeonia import (
    Bar, Pitch, Score, Staff, Tonality, TonalityPlan, Voice,
)

## 1. Enharmonic pitches

Spelling is structural even when two pitches have the same MIDI value.

In [ ]:
d_sharp = Pitch.parse("D#4")
e_flat = Pitch.parse("Eb4")
d_sharp, e_flat, d_sharp == e_flat, d_sharp.enharmonic_equals(e_flat)

## 2. A bar with source tonality

In [ ]:
c_major = Tonality("C", "major")
source_bar = Bar("C E G B", tonality=c_major)
source_bar

## 3. Same-tonic mode change

`apply_tonality` retains scale degrees while realizing them in a new mode.

In [ ]:
c_dorian = Tonality("C", "dorian")
c_dorian_bar = source_bar.apply_tonality(c_dorian)
c_dorian_bar

## 4. Tonic-and-mode change

In [ ]:
d_dorian = Tonality("D", "dorian")
d_dorian_bar = source_bar.apply_tonality(d_dorian)
d_dorian_bar

## 5. Chromatic alteration preservation

The altered fourth in C major remains an altered fourth when mapped to D major.

In [ ]:
altered = Bar("C F# G", tonality=c_major)
altered_in_d = altered.apply_tonality(Tonality("D", "major"))
altered, altered_in_d

## 6. A voice with a persistent tonal plan

In [ ]:
g_major = Tonality("G", "major")
d_major = Tonality("D", "major")
voice = Voice(
    [Bar("C E G"), Bar("G B D"), Bar("D F# A")],
    default_tonality=c_major,
    tonality_plan=TonalityPlan.from_mapping({1: g_major, 2: d_major}),
    name="Melody",
)
[voice.tonality_at(index) for index in range(len(voice))]

## 7. Two staves with a global modulation plan

In [ ]:
score = Score(
    default_tonality=c_major,
    tonality_plan={1: g_major, 2: d_major},
    tempo=108,
    time_signature=(3, 4),
    title="Tonal plan",
)
score["melody"] = Staff(
    Voice([Bar("C E G"), Bar("G B D"), Bar("D F# A")]),
    name="Melody", midi_channel=0, midi_program=40,
)
score["bass"] = Staff(
    Voice([Bar("C, G, C"), Bar("G, D G"), Bar("D, A, D")]),
    clef="bass", name="Bass", midi_channel=1, midi_program=42,
)
[score.tonality_at("melody", index) for index in range(3)]

## 8. Triadex status

`TriadexMuse` was intentionally removed from the current package, so this workbench does not contain a non-runnable import. A future generator can feed ordinary `Bar` values into the same `Voice` and `Score` workflow.

## 9. Rendering and playback

LilyPond strings and MIDI files need no external programs. Set either opt-in flag below to display notation with LilyPond or preview audio with FluidSynth when the corresponding executable is installed.

In [ ]:
from shutil import which

lilypond_source = score.to_lilypond()
print(lilypond_source)

RENDER_WITH_LILYPOND = False
PLAY_WITH_FLUIDSYNTH = False

if RENDER_WITH_LILYPOND:
    if which("lilypond") is None:
        raise FileNotFoundError("Install LilyPond and add it to PATH")
    score.show()

if PLAY_WITH_FLUIDSYNTH:
    if which("fluidsynth") is None:
        raise FileNotFoundError("Install FluidSynth and add it to PATH")
    score.play()